# 07 — Loss Function Experiments: XGBoost Poisson & CatBoost Poisson

**Prerequisite**: `notebooks/04_training.ipynb` must be complete.  
These files must exist:
- `data/processed/zone_hour_grid.parquet`
- `data/processed/cis_table.parquet`
- `checkpoints/catboost_hour_*/` (current winner)
- `data/outputs/eval_*.json` (baseline results for comparison)

**What this notebook does**:
1. Runs two targeted ablations defined in `configs/model.yaml` → `# Experiment Variants`:
   - `xgboost_poisson`: XGBoost with `count:poisson` loss (vs default `reg:squarederror`)
   - `catboost_poisson`: CatBoost with `loss_function: Poisson` (vs default `RMSE`)
2. Uses the EXACT same train/test split, features, and zone aggregates as `04_training.ipynb`
3. Compares results against all 3 standard models (from `eval_*.json`)
4. Determines whether Poisson loss improves MAE/RMSE and per-hour NDCG
5. Makes a final winner recommendation

**Rationale (from session_log.md PLAN → Action 2.1 / CB-1)**:  
RMSE/MAE ratio = 2.37 in the current best model confirms that MSE-based loss is
penalising rare high-count spikes by shrinking predictions toward the mean.  
Poisson loss uses a log-link function natively suited to right-skewed count distributions.

**Files saved**:
- `checkpoints/xgboost_poisson_hour_{timestamp}/`
- `checkpoints/catboost_poisson_hour_{timestamp}/`
- `data/outputs/experiment_{timestamp}.json`

## Cell 1 — Environment setup
**Expected output**: `Project root: ...GridLock_R2_Transfer`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\USER\Desktop\GridLock_R2_Transfer


## Cell 2 — Configure loguru
**Expected output**: `Loguru configured.`

In [2]:
import sys
from loguru import logger

logger.remove()
logger.add(
    sys.stdout,
    format='<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}',
    level='INFO',
    colorize=False,
)
print('Loguru configured.')

Loguru configured.


## Cell 3 — Verify prerequisite files
**What this cell does**: Check that all data files required for experiments exist.  
**Expected output**: All files found. No `MISSING` lines.

In [3]:
import glob

required_files = [
    PROJECT_ROOT / 'data' / 'processed' / 'zone_hour_grid.parquet',
    PROJECT_ROOT / 'data' / 'processed' / 'cis_table.parquet',
    PROJECT_ROOT / 'configs' / 'model.yaml',
    PROJECT_ROOT / 'configs' / 'features.yaml',
    PROJECT_ROOT / 'configs' / 'eval.yaml',
]

all_ok = True
for f in required_files:
    if f.exists():
        print(f'  ✓  {f.name}  ({f.stat().st_size / 1e3:.1f} KB)')
    else:
        print(f'  ✗  MISSING: {f.name} — run 04_training.ipynb first!')
        all_ok = False

# Check baseline eval JSON
eval_files = sorted(glob.glob(str(PROJECT_ROOT / 'data' / 'outputs' / 'eval_*.json')))
if eval_files:
    print(f'  ✓  baseline eval: {Path(eval_files[-1]).name}')
else:
    print('  ✗  No baseline eval_*.json found — comparison will be empty')

if not all_ok:
    raise FileNotFoundError('One or more prerequisite files are missing. See above.')
print('\nAll prerequisite files found. Proceed.')

  ✓  zone_hour_grid.parquet  (226.7 KB)
  ✓  cis_table.parquet  (10.7 KB)
  ✓  model.yaml  (10.7 KB)
  ✓  features.yaml  (9.5 KB)
  ✓  eval.yaml  (9.1 KB)
  ✓  baseline eval: eval_20260619_155555.json

All prerequisite files found. Proceed.


## Cell 4 — Preview experiment variants from model.yaml
**What this cell does**: Print the experiment variant configs so you can verify them before running.  
**Expected output**: Two variant blocks — `xgboost_poisson` and `catboost_poisson` — showing their overrides.

In [4]:
import yaml

with open(PROJECT_ROOT / 'configs' / 'model.yaml') as f:
    model_cfg = yaml.safe_load(f)

variants_to_run = ['xgboost_poisson', 'catboost_poisson']

print(f'Current primary model: {model_cfg.get("primary_model")}')
print(f'Current winner MAE:    {model_cfg.get("winner_mae")}')
print(f'Current winner RMSE:   {model_cfg.get("winner_rmse")}')
print()

for v in variants_to_run:
    cfg = model_cfg.get(v, {})
    if not cfg:
        print(f'  ✗  {v}: NOT FOUND in model.yaml — check configs/model.yaml')
        continue
    print(f'  ✓  {v}:')
    print(f'       base    : {cfg.get("base")}')
    print(f'       changes : {cfg.get("changes")}')
    print(f'       rationale: {cfg.get("expected_impact")}')
    print()

Current primary model: catboost
Current winner MAE:    4.5863
Current winner RMSE:   10.1618

  ✓  xgboost_poisson:
       base    : xgboost
       changes : {'objective': 'count:poisson', 'eval_metric': 'rmse'}
       rationale: RMSE reduction 5-15% (S1, S5 findings)

  ✓  catboost_poisson:
       base    : catboost
       changes : {'loss_function': 'Poisson'}
       rationale: RMSE reduction 5-15% (same rationale as xgboost_poisson)



## Cell 5 — Run experiments
**What this cell does**: Calls `src.training.experiment.run_experiments()` which:
- Trains `xgboost_poisson` and `catboost_poisson` on `zone_hour_grid.parquet`
- Uses the IDENTICAL split, features, and zone aggregates as the standard 04 training run
- Evaluates all metrics: MAE, RMSE, per-hour NDCG, Spearman ρ, PAI
- Compares against the 3 standard baseline models from `eval_*.json`
- Saves checkpoints for both variants

**Expected runtime**: ~3–6 minutes (XGBoost ~60s, CatBoost ~3–4 min)

**Expected output**: Training logs with SCORECARD at the end showing all models ranked.

In [5]:
from src.training.experiment import run_experiments

results = run_experiments(
    project_root=PROJECT_ROOT,
    variants=['xgboost_poisson', 'catboost_poisson'],
    time_resolution='hour',
)

print('\nExperiment run complete.')

22:18:17 | INFO     | Configs loaded | features.yaml hash: e2322218c95b43af...
22:18:17 | INFO     | 
  EXPERIMENT RUN | timestamp=20260619_164817
  Variants: ['xgboost_poisson', 'catboost_poisson']
  Resolution: hour
22:18:17 | INFO     | CIS table loaded: 140 zones
22:18:17 | INFO     | Grid loaded: 26,354 rows from 'zone_hour_grid.parquet'
22:18:17 | INFO     | Split: train=19,870 rows (2023-11-09 → 2024-02-29) | test=6,484 rows (2024-03-01 → 2024-04-08) | Leakage guard: PASSED ✓
22:18:17 | INFO     | Computing zone aggregate features (train-only) ...
22:18:17 | INFO     | Zone aggregates computed for 140 training zones | zone_mean_count range: [1.10, 65.32]
22:18:17 | INFO     | X_train: (19870, 18)  y_train mean=10.30 max=284
X_val:   (6484, 18)    y_val mean=9.82 max=266


Experiment variants:   0%|          | 0/2 [00:00<?, ?variant/s]

22:18:17 | INFO     | 
────────────────────────────────────────────────────────────
  Experiment: xgboost_poisson
────────────────────────────────────────────────────────────
22:18:17 | INFO     | Variant 'xgboost_poisson': base='xgboost' | overrides={'objective': 'count:poisson', 'eval_metric': 'rmse'}



Fitting xgboost_poisson:   0%|          | 0/1 [00:00<?, ?model/s]

22:18:19 | INFO     |   XGBoost early-stop: best_round=118 | final_val_rmse=10.7787


...

22:18:35 | INFO     | === Evaluating: catboost_poisson | hour | target='zone_hour_violation_count' ===
22:18:35 | INFO     | [catboost_poisson/hour] MAE=4.8431  RMSE=10.4792
22:18:35 | INFO     | Naive mean-per-zone baseline: MAE=6.9714  RMSE=15.8952
22:18:35 | INFO     |   BEATS naive baseline: MAE 4.8431 vs 6.9714 (+30.5% improvement)
22:18:35 | INFO     | Relevance assigned: grade=2 → 35 zones | grade=1 → 35 zones | grade=0 → 67 zones (q50=36.0, q75=113.0)
22:18:35 | INFO     |   [catboost_poisson] NDCG@5=1.0000  Precision@5=1.0000
22:18:35 | INFO     |   [catboost_poisson] NDCG@10=1.0000  Precision@10=1.0000
22:18:35 | INFO     | Frequency baseline: 140 zones | top-3 zones: {2: 185578.5, 7: 1537.732546, 4: 160.647921}
22:18:35 | INFO     |   [freq-baseline] NDCG@5=1.0000  Precision@5=1.0000
22:18:35 | INFO     |   [freq-baseline] NDCG@10=1.0000  Precision@10=1.0000
22:18:35 | WARNING  |   [catboost_poisson] does NOT beat freq ranker on NDCG.
22:18:35 | INFO     |   [catboost_poisso

C:\Users\USER\Desktop\GridLock_R2_Transfer\src\evaluation\metrics.py:716: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(zone_true.loc[common].values, zone_pred_priority.values)
C:\Users\USER\Desktop\GridLock_R2_Transfer\src\evaluation\metrics.py:716: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(zone_true.loc[common].values, zone_pred_priority.values)


22:18:41 | INFO     | Temporal Spearman ρ: mean=0.4938 std=0.2805 (n_hours=579)
22:18:41 | INFO     | Relevance assigned: grade=2 → 5 zones | grade=1 → 3 zones | grade=0 → 6 zones (q50=3.0, q75=4.0)
22:18:41 | INFO     | Relevance assigned: grade=2 → 5 zones | grade=1 → 3 zones | grade=0 → 5 zones (q50=2.0, q75=4.0)
22:18:41 | INFO     | Relevance assigned: grade=2 → 3 zones | grade=1 → 2 zones | grade=0 → 4 zones (q50=2.0, q75=3.0)
22:18:41 | INFO     | Relevance assigned: grade=2 → 2 zones | grade=1 → 3 zones | grade=0 → 2 zones (q50=2.0, q75=4.0)
22:18:41 | INFO     | Relevance assigned: grade=2 → 3 zones | grade=1 → 2 zones | grade=0 → 5 zones (q50=1.5, q75=5.8)
22:18:41 | INFO     | Relevance assigned: grade=2 → 4 zones | grade=1 → 3 zones | grade=0 → 6 zones (q50=6.0, q75=9.0)
22:18:41 | INFO     | Relevance assigned: grade=2 → 5 zones | grade=1 → 5 zones | grade=0 → 9 zones (q50=3.0, q75=4.5)
22:18:41 | INFO     | Relevance assigned: grade=2 → 4 zones | grade=1 → 1 zones | grade

Experiment variants: 100%|██████████| 2/2 [00:32<00:00, 16.10s/variant]

22:18:49 | INFO     | Baseline results loaded from: 'eval_20260619_155555.json'
22:18:49 | INFO     | Experiment results saved → 'C:\Users\USER\Desktop\GridLock_R2_Transfer\data\outputs\experiment_20260619_164817.json'
22:18:49 | INFO     | 
22:18:49 | INFO     |   EXPERIMENT SCORECARD
22:18:49 | INFO     | ================================================================================
22:18:49 | INFO     |   Model                              MAE     RMSE   NDCG/hr   Spear    PAI  Beats BL
22:18:49 | INFO     |   ------------------------------------------------------------------------------
22:18:49 | INFO     |   xgboost_hour                          4.5892  10.1345    0.8896  0.5143  11.19         ✓
22:18:49 | INFO     |   catboost_hour                         4.5863  10.1618    0.8888  0.5123  11.19         ✓
22:18:49 | INFO     |   xgboost_poisson                [EXP]  4.7308  10.7541    0.8888  0.5085  11.19         ✓
22:18:49 | INFO     |   lightgbm_hour                        

## Cell 6 — Print structured comparison table
**What this cell does**: Extract and display the comparison table from results as a DataFrame.  
**Expected output**: Table of all models (experiments + baselines) sorted by per-hour NDCG.

In [6]:
import pandas as pd

table = results.get('baseline_comparison', [])
if not table:
    print('No comparison table found — check experiment output.')
else:
    df = pd.DataFrame(table)
    display_cols = ['name', 'is_experiment', 'overrides', 'mae', 'rmse',
                    'per_hour_ndcg', 'per_hour_spear', 'pai', 'beats_baseline_per_hour']
    display_cols = [c for c in display_cols if c in df.columns]
    
    print('=== FULL COMPARISON TABLE (sorted by per-hour NDCG desc) ===')
    pd.set_option('display.max_colwidth', 40)
    pd.set_option('display.float_format', '{:.4f}'.format)
    display(df[display_cols])
    
    # Highlight winners
    print()
    best_overall = df.iloc[0]
    best_exp = df[df['is_experiment'] == True].iloc[0] if df['is_experiment'].any() else None
    
    print(f'Best overall      : {best_overall["name"]} (per-hour NDCG={best_overall["per_hour_ndcg"]:.4f}, MAE={best_overall["mae"]:.4f})')
    if best_exp is not None:
        print(f'Best experiment   : {best_exp["name"]} (per-hour NDCG={best_exp["per_hour_ndcg"]:.4f}, MAE={best_exp["mae"]:.4f})')

=== FULL COMPARISON TABLE (sorted by per-hour NDCG desc) ===


,name,is_experiment,overrides,mae,rmse,per_hour_ndcg,per_hour_spear,pai,beats_baseline_per_hour
0,xgboost_hour,False,{},4.5892,10.1345,0.8896,0.5143,11.1900,True
1,catboost_hour,False,{},4.5863,10.1618,0.8888,0.5123,11.1900,True
2,xgboost_poisson,True,"{'objective': 'count:poisson', 'eval...",4.7308,10.7541,0.8888,0.5085,11.1900,True
3,lightgbm_hour,False,{},4.6741,10.2433,0.8864,0.5012,11.1900,True
4,catboost_poisson,True,{'loss_function': 'Poisson'},4.8431,10.4792,0.8839,0.4938,11.1900,True



Best overall      : xgboost_hour (per-hour NDCG=0.8896, MAE=4.5892)
Best experiment   : xgboost_poisson (per-hour NDCG=0.8888, MAE=4.7308)


## Cell 7 — Detailed metric comparison: Poisson vs RMSE
**What this cell does**: Side-by-side comparison of XGBoost RMSE vs Poisson, and CatBoost RMSE vs Poisson.  
**Expected output**: Delta table showing which metrics improved, which worsened, and by how much.

In [7]:
import json
import glob

# Load baseline results
baseline_file = sorted(glob.glob(str(PROJECT_ROOT / 'data' / 'outputs' / 'eval_*.json')))[-1]
with open(baseline_file) as f:
    baseline = json.load(f)

def extract_metrics(result: dict) -> dict:
    reg   = result.get('regression', {})
    rph   = result.get('ranking_per_hour', {})
    ndcg  = rph.get('model_ndcg', {})
    spear = rph.get('model_spearman', {})
    pai   = result.get('spatial_pai', {})
    return {
        'MAE':          round(reg.get('mae', float('nan')), 4),
        'RMSE':         round(reg.get('rmse', float('nan')), 4),
        'NDCG/hr':      round(ndcg.get('mean_ndcg', float('nan')), 4),
        'Spearman':     round(spear.get('mean_spearman', float('nan')), 4),
        'PAI':          round(pai.get('pai', float('nan')), 2),
        'Beats BL NDCG': rph.get('beats_baseline_per_hour_ndcg', False),
    }

rows = {}

# XGBoost RMSE (baseline)
if 'xgboost_hour' in baseline:
    rows['xgboost_RMSE (baseline)'] = extract_metrics(baseline['xgboost_hour'])

# XGBoost Poisson (experiment)
if 'xgboost_poisson' in results:
    rows['xgboost_Poisson [EXP]'] = extract_metrics(results['xgboost_poisson'])

print('=== XGBoost: RMSE Loss vs Poisson Loss ===')
xgb_rows = {k: v for k, v in rows.items() if 'xgboost' in k}
if xgb_rows:
    display(pd.DataFrame(xgb_rows).T)

# CatBoost RMSE (baseline)
rows2 = {}
if 'catboost_hour' in baseline:
    rows2['catboost_RMSE (baseline)'] = extract_metrics(baseline['catboost_hour'])

# CatBoost Poisson (experiment)
if 'catboost_poisson' in results:
    rows2['catboost_Poisson [EXP]'] = extract_metrics(results['catboost_poisson'])

print()
print('=== CatBoost: RMSE Loss vs Poisson Loss ===')
if rows2:
    display(pd.DataFrame(rows2).T)

=== XGBoost: RMSE Loss vs Poisson Loss ===


,MAE,RMSE,NDCG/hr,Spearman,PAI,Beats BL NDCG
xgboost_RMSE (baseline),4.5892,10.1345,0.8896,0.5143,11.1900,True
xgboost_Poisson [EXP],4.7308,10.7541,0.8888,0.5085,11.1900,True



=== CatBoost: RMSE Loss vs Poisson Loss ===


,MAE,RMSE,NDCG/hr,Spearman,PAI,Beats BL NDCG
catboost_RMSE (baseline),4.5863,10.1618,0.8888,0.5123,11.1900,True
catboost_Poisson [EXP],4.8431,10.4792,0.8839,0.4938,11.1900,True


## Cell 8 — Decision: update model.yaml if a Poisson variant wins
**What this cell does**: Evaluate whether any Poisson variant beats the current winner (CatBoost RMSE).  
If yes, prints the config update command. If no, confirms the current winner is unchanged.  
**Expected output**: A decision statement with metric evidence.

In [8]:
import yaml

# Current winner metrics
current_winner_mae  = model_cfg.get('winner_mae', 4.5863)
current_winner_rmse = model_cfg.get('winner_rmse', 10.1618)
current_winner_name = model_cfg.get('primary_model', 'catboost')
current_winner_ndcg = model_cfg.get('winner_per_hour_ndcg_mean', 0.8888)

print(f'Current winner: {current_winner_name}')
print(f'  MAE  = {current_winner_mae}')
print(f'  RMSE = {current_winner_rmse}')
print(f'  Per-hour NDCG = {current_winner_ndcg}')
print()

best_exp_name = None
best_exp_mae  = float('inf')
best_exp_rmse = float('inf')
best_exp_ndcg = 0.0

for variant in ['xgboost_poisson', 'catboost_poisson']:
    if variant not in results:
        continue
    m = extract_metrics(results[variant])
    print(f'{variant}: MAE={m["MAE"]}, RMSE={m["RMSE"]}, NDCG/hr={m["NDCG/hr"]}')
    if m['MAE'] < best_exp_mae:
        best_exp_mae  = m['MAE']
        best_exp_rmse = m['RMSE']
        best_exp_ndcg = m['NDCG/hr']
        best_exp_name = variant

print()

# Decision criterion: per-hour NDCG is primary; MAE is tiebreaker (per plan)
if best_exp_name and (best_exp_ndcg > current_winner_ndcg or
                      (abs(best_exp_ndcg - current_winner_ndcg) < 0.001 and best_exp_mae < current_winner_mae)):
    print(f'✅ EXPERIMENT WINS: {best_exp_name}')
    print(f'   MAE  {current_winner_mae} → {best_exp_mae} (Δ = {best_exp_mae - current_winner_mae:+.4f})')
    print(f'   RMSE {current_winner_rmse} → {best_exp_rmse} (Δ = {best_exp_rmse - current_winner_rmse:+.4f})')
    print(f'   NDCG {current_winner_ndcg} → {best_exp_ndcg} (Δ = {best_exp_ndcg - current_winner_ndcg:+.4f})')
    print()
    print('ACTION: Update configs/model.yaml:')
    print(f'  primary_model  → "{best_exp_name.replace("_poisson", "")}"')
    print(f'  winner_mae     → {best_exp_mae}')
    print(f'  winner_rmse    → {best_exp_rmse}')
    print(f'  winner_per_hour_ndcg_mean → {best_exp_ndcg}')
    print(f'  (and update the loss_function in the model block to Poisson)')
else:
    print(f'✅ CURRENT WINNER HOLDS: {current_winner_name} (RMSE loss)')
    if best_exp_name:
        print(f'   Best experiment ({best_exp_name}) did NOT improve over current winner.')
        print(f'   Current: MAE={current_winner_mae}, RMSE={current_winner_rmse}, NDCG={current_winner_ndcg}')
        print(f'   Experiment: MAE={best_exp_mae}, RMSE={best_exp_rmse}, NDCG={best_exp_ndcg}')
    print()
    print('ACTION: No config changes needed. Poisson loss variants do not improve performance.')

Current winner: catboost
  MAE  = 4.5863
  RMSE = 10.1618
  Per-hour NDCG = 0.8888

xgboost_poisson: MAE=4.7308, RMSE=10.7541, NDCG/hr=0.8888
catboost_poisson: MAE=4.8431, RMSE=10.4792, NDCG/hr=0.8839

✅ CURRENT WINNER HOLDS: catboost (RMSE loss)
   Best experiment (xgboost_poisson) did NOT improve over current winner.
   Current: MAE=4.5863, RMSE=10.1618, NDCG=0.8888
   Experiment: MAE=4.7308, RMSE=10.7541, NDCG=0.8888

ACTION: No config changes needed. Poisson loss variants do not improve performance.


## Cell 9 — Summary
**What this cell does**: Print a final summary of what was done, what was saved, and what the next step is.  
**Expected output**: Path to experiment JSON, paths to new checkpoints, and next recommended action.

In [9]:
import glob

ts = results.get('experiment_timestamp', 'unknown')

print('=' * 60)
print('  EXPERIMENT SUMMARY')
print('=' * 60)
print()
print('What was done:')
print('  - Trained xgboost_poisson (count:poisson objective)')
print('  - Trained catboost_poisson (loss_function: Poisson)')
print('  - Compared both against catboost_hour baseline (current winner)')
print()
print('Files saved:')
exp_json = PROJECT_ROOT / 'data' / 'outputs' / f'experiment_{ts}.json'
print(f'  - Experiment results: {exp_json}')
for v in results.get('variants_run', []):
    ckpt_dirs = sorted(glob.glob(str(PROJECT_ROOT / 'checkpoints' / f'{v}_hour_{ts}')))
    for d in ckpt_dirs:
        print(f'  - Checkpoint: {Path(d).name}')
print()
print('What the experiment found:')
table = results.get('baseline_comparison', [])
if table:
    best = table[0]
    print(f'  - Best model: {best["name"]} (per-hour NDCG={best["per_hour_ndcg"]:.4f}, MAE={best["mae"]:.4f})')
print()
print('Next step:')
print('  → If a Poisson variant won: update model.yaml and re-run src/inference/ranker.py')
print('  → If current winner holds: proceed to Action 2.3 — add lag_1d_count + lag_7d_count')
print('  → Run 06_shap.ipynb on the CatBoost checkpoint (CB-3 from MODEL UPDATE)')

  EXPERIMENT SUMMARY

What was done:
  - Trained xgboost_poisson (count:poisson objective)
  - Trained catboost_poisson (loss_function: Poisson)
  - Compared both against catboost_hour baseline (current winner)

Files saved:
  - Experiment results: C:\Users\USER\Desktop\GridLock_R2_Transfer\data\outputs\experiment_20260619_164817.json
  - Checkpoint: xgboost_poisson_hour_20260619_164817
  - Checkpoint: catboost_poisson_hour_20260619_164817

What the experiment found:
  - Best model: xgboost_hour (per-hour NDCG=0.8896, MAE=4.5892)

Next step:
  → If a Poisson variant won: update model.yaml and re-run src/inference/ranker.py
  → If current winner holds: proceed to Action 2.3 — add lag_1d_count + lag_7d_count
  → Run 06_shap.ipynb on the CatBoost checkpoint (CB-3 from MODEL UPDATE)
